# Load a Model from Kaggle

This tutorial shows how to download and load pretrained models from the Kaggle Model Hub into Julia using `ONNX.jl` and `Flux.jl`.

By the end you will know how to:
- Install the required packages
- Download a Kaggle model archive
- Load an ONNX model and inspect its graph
- Load a Flux model from saved weights
- Save and reload from a local directory

## 1. Install Packages

If you have not installed these packages yet, run the cell below once. It is safe to re-run.

In [ ]:
using Pkg
Pkg.add(["ONNX", "Flux", "HTTP", "JSON3"])

## 2. Download a Model from Kaggle

Kaggle model files are hosted at `https://www.kaggle.com/models/{owner}/{model}/frameworks/{framework}/variations/{variation}/versions/{version}`.

The simplest way to obtain them is with the Kaggle CLI or direct HTTP download. Below we use the Kaggle REST API to fetch the download URL and stream the file.

In [ ]:
using HTTP

# Set your Kaggle credentials (from https://www.kaggle.com/settings — API section)
kaggle_username = ENV["KAGGLE_USERNAME"]   # or replace with a string
kaggle_key      = ENV["KAGGLE_KEY"]

# Example: download ResNet-50 ONNX weights from Kaggle
url = "https://www.kaggle.com/api/v1/models/keras/resnet/keras/resnet50/2/download"

resp = HTTP.get(url;
    headers = ["Authorization" => "Basic " * base64encode("$kaggle_username:$kaggle_key")],
)

model_path = "resnet50.onnx"
write(model_path, resp.body)
println("Downloaded ", filesize(model_path) ÷ 1024, " KB → ", model_path)

## 3. Load an ONNX Model

`ONNX.jl` reads any `.onnx` file and returns a callable Flux chain. The model can be used directly for inference without any additional configuration.

In [ ]:
using ONNX

model = ONNX.load(model_path)

println("Model type:    ", typeof(model))
println("Input names:   ", ONNX.input_names(model))
println("Output names:  ", ONNX.output_names(model))
println("Operator count: ", length(model.graph.node))

## 4. Supported ONNX Frameworks on Kaggle

Kaggle hosts models in several framework formats. The table below shows which Julia package to use for each.

| Kaggle framework | File extension | Julia package |
|---|---|---|
| ONNX | `.onnx` | `ONNX.jl` |
| TensorFlow / Keras (SavedModel) | `saved_model.pb` | `ONNX.jl` (after export) |
| PyTorch (via ONNX export) | `.onnx` | `ONNX.jl` |
| Flux.jl | `.bson` / `.jld2` | `BSON.jl` / `JLD2.jl` |
| scikit-learn (via ONNX export) | `.onnx` | `ONNX.jl` |

Most production models on Kaggle are available as ONNX exports, which is the recommended format for cross-framework interoperability.

## 5. Inspect the ONNX Graph

Before running inference it is useful to inspect the model's input and output shapes so you know what tensor format to prepare.

In [ ]:
# Inspect input shape (ONNX stores shapes as symbolic dimensions)
for inp in model.graph.input
    dims = [d.dim_value for d in inp.type.tensor_type.shape.dim]
    println("Input  '" , inp.name, "'  shape: ", dims)
end

for out in model.graph.output
    dims = [d.dim_value for d in out.type.tensor_type.shape.dim]
    println("Output '", out.name, "'  shape: ", dims)
end

## 6. Load a Flux Model from BSON

Kaggle also hosts models saved directly from Flux. These are stored as `.bson` files and can be reloaded with `BSON.jl`.

In [ ]:
using Pkg; Pkg.add("BSON")
using BSON: @load

# Replace with the path to your downloaded .bson file
@load "flux_model.bson" model

println("Loaded Flux model: ", typeof(model))
println("Parameters:        ", sum(length, Flux.params(model)))

## 7. Load a Flux Model from JLD2

`JLD2.jl` is an alternative serialisation format that is faster to read and supports arbitrary Julia types.

In [ ]:
using Pkg; Pkg.add("JLD2")
using JLD2

# Replace with the path to your downloaded .jld2 file
model_jld2 = JLD2.load("flux_model.jld2", "model")

println("Loaded model: ", typeof(model_jld2))

## 8. Save and Reload an ONNX Model Locally

Once downloaded, cache the file locally to avoid re-downloading on every run.

In [ ]:
cache_dir = joinpath(homedir(), ".cache", "kaggle", "models")
mkpath(cache_dir)

cached_path = joinpath(cache_dir, "resnet50.onnx")

if !isfile(cached_path)
    cp(model_path, cached_path)
    println("Cached to: ", cached_path)
else
    println("Already cached: ", cached_path)
end

# Reload from cache
model_cached = ONNX.load(cached_path)
println("Reloaded from cache: ", typeof(model_cached))

## 9. Query the Kaggle Model API

You can search for models programmatically using the Kaggle REST API before downloading.

In [ ]:
using HTTP, JSON3, Base64

function kaggle_search(query::String; page_size=5)
    url  = "https://www.kaggle.com/api/v1/models/list?search=$(HTTP.escapeuri(query))&pageSize=$page_size&sortBy=hotness"
    resp = HTTP.get(url;
        headers = ["Authorization" => "Basic " * base64encode("$kaggle_username:$kaggle_key")],
    )
    data = JSON3.read(resp.body)
    return data.models
end

results = kaggle_search("resnet image classification")
for m in results
    println(rpad(m.ref, 50), "  votes: ", m.voteCount)
end

## Summary

| Task | How |
|---|---|
| Download a model | `HTTP.get` with Kaggle Basic Auth |
| Load ONNX model | `ONNX.load(path)` |
| Inspect inputs/outputs | `ONNX.input_names`, `ONNX.output_names` |
| Load Flux model (BSON) | `BSON.@load path model` |
| Load Flux model (JLD2) | `JLD2.load(path, "model")` |
| Cache locally | `cp` to `~/.cache/kaggle/models/` |
| Search models via API | `HTTP.get` Kaggle `/api/v1/models/list` |

Once the model is loaded, continue with **Tutorial 02 — Run Inference** to put it to work.